### Linear SVC (Linear Support Vector Classification)

In [44]:
import pandas as pd
import numpy as np
import time
import warnings
import tracemalloc
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from scipy.stats import loguniform

In [45]:
def linear_svc(data):
    y = data['collision']
    x = data.drop('collision', axis=1)
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=81)

    svc = LinearSVC(random_state=81, dual=False, max_iter=10_000)

    tracemalloc.start()
    start_time = time.time()

    svc.fit(x_train, y_train)

    training_time = time.time() - start_time
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    y_pred = svc.predict(x_test)
    
    f1_metric = f1_score(y_test, y_pred)
    peak_memory_mb = peak_memory / (1024 * 1024)

    return f1_metric, training_time, peak_memory_mb

#### Mетрики без подбора гиперпараметров

In [46]:
n_samples = [100, 500, 1000, 3000]
m_features = [5, 8, 11]

results = []

for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv")
        f1, real_time, mem_usage = linear_svc(data)
        results.append({
            'samples (n)' : n,
            'features (m)' : m,
            'f1-score' : f1,
            'time (sec)' : real_time,
            'memory (MB)': mem_usage
        })

results_df = pd.DataFrame(results)

f1_avg = results_df['f1-score'].mean()
real_time_avg = results_df['time (sec)'].mean()
mem_usage_avg = results_df['memory (MB)'].mean()

print(f'Средняя f1-мера = {round(f1_avg, 3)}')
print(f'Среднее время = {round(real_time_avg, 5)} сек')
print(f'Среднее потребление памяти = {round(mem_usage_avg, 3)} MB')

results_df

Средняя f1-мера = 0.936
Среднее время = 0.00398 сек
Среднее потребление памяти = 0.166 MB


,samples (n),features (m),f1-score,time (sec),memory (MB)
0,100,5,1.000000,0.003748,0.015041
1,100,8,0.896552,0.003452,0.014973
2,100,11,0.838710,0.002972,0.026678
3,500,5,0.986842,0.002571,0.045798
4,500,8,0.936508,0.002936,0.067167
5,500,11,0.886076,0.003326,0.109594
6,1000,5,0.996364,0.002653,0.088130
7,1000,8,0.981550,0.002835,0.129957
8,1000,11,0.882716,0.004433,0.216040
9,3000,5,0.996328,0.003968,0.247724


Метод `Linear SVC` показывает высокую эффективность в решении поставленной задачи, о чем говорит высокий `f1-score` и низкая затрата памяти `time`. F-мера позволяет сбалансировано оценить точность (Precision) и полноту (Recall) модели. 

На самых маленьких наборах (100 строк) F-мера доходит до 1.0. Это происходит потому, что данных мало и модель легко находит в них идеальные закономерности. С ростом данных до 3000 строк F-мера немного снижается (до 0.86), так как данных становится больше и они становятся сложнее

#### Mетрики с подбором гиперпараметров

In [47]:
def linear_svc_params(data):
    y = data['collision']
    x = data.drop('collision', axis=1)
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=81)

    param_dist = {
        'C': loguniform(1e-3, 1e2),     # регуляризация (штраф за ошибки)
        'loss': ['hinge', 'squared_hinge']      # ф-ия потерь
    }

    svc = LinearSVC(random_state=81, max_iter=10_000)

    random_search = RandomizedSearchCV(
        svc, param_dist, n_iter=10, cv=5, scoring='f1', random_state=81
    )

    tracemalloc.start()
    start_time = time.time()

    random_search.fit(x_train, y_train)

    training_time = time.time() - start_time
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(x_test)
    
    f1_metric = f1_score(y_test, y_pred)
    best_params = random_search.best_params_
    peak_memory_mb = peak_memory / (1024 * 1024)

    return f1_metric, training_time, peak_memory_mb, best_params

In [49]:
results = []

for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv")
        f1, real_time, mem_usage, best_p = linear_svc_params(data)
        results.append({
            'samples (n)' : n,
            'features (m)' : m,
            'f1-score' : f1,
            'time (sec)' : real_time,
            'memory (MB)': mem_usage,
            'best C': best_p['C'],
            'best loss': best_p['loss']
        })

results_df = pd.DataFrame(results)

f1_avg = results_df['f1-score'].mean()
real_time_avg = results_df['time (sec)'].mean()
mem_usage_avg = results_df['memory (MB)'].mean()

print(f'Средняя f1-мера = {round(f1_avg, 3)}')
print(f'Среднее время = {round(real_time_avg, 5)} сек')
print(f'Среднее потребление памяти = {round(mem_usage_avg, 3)} MB')

results_df

Средняя f1-мера = 0.933
Среднее время = 1.35966 сек
Среднее потребление памяти = 1.466 MB


,samples (n),features (m),f1-score,time (sec),memory (MB),best C,best loss
0,100,5,0.960000,0.377436,13.504023,12.675095,squared_hinge
1,100,8,0.896552,0.315163,0.113337,1.076415,squared_hinge
2,100,11,0.838710,0.454860,0.091887,12.675095,squared_hinge
3,500,5,0.993377,0.567050,0.141665,12.675095,squared_hinge
4,500,8,0.936508,0.691474,0.168582,12.675095,squared_hinge
5,500,11,0.886076,1.100721,0.228560,37.076261,squared_hinge
6,1000,5,0.996364,0.876255,0.228406,12.675095,squared_hinge
7,1000,8,0.977941,0.876987,0.284133,37.076261,squared_hinge
8,1000,11,0.886154,1.859199,0.406859,12.675095,squared_hinge
9,3000,5,0.996319,1.740578,0.562342,12.675095,squared_hinge


После проведения автоматизированного подбора гиперпараметров с помощью `RandomizedSearchCV`, значение F-меры составило **0.933** (ранее - **0.936**). Незначительное снижение метрики объясняется применением кросс-валидации по 5 блокам, что обеспечивает более объективную оценку обобщающей способности модели и снижает риск переобучения, характерный для одиночного разбиения данных.


Для оптимизации модели использовалось пространство поиска параметров `C` и `loss`.
- Параметр `C` выбирался из логарифмического распределения в диапазоне от 0.001 до 100, что позволило эффективно протестировать как "мягкую", так и "строгую" регуляризацию.
- Выбор между функциями потерь `hinge` и `squared_hinge` позволил алгоритму определить наиболее эффективный способ математического наказания за ошибки классификации для данного набора данных